
# Recruitment Regression Template

This notebook is a **template** for running the pair-specific Illinois-vs-control regression for the **Recruitment** analytic sample.

## What this notebook assumes

- You are running a regression for **one adjacent year pair at a time**
- Each row in the file already represents a **transition from year t to year t+1**
- The dependent variable is already constructed in the analytic sample:
  - `joined_state_local`
- The key treatment indicator is:
  - `illinois` = 1 for Illinois, 0 for control states

## Interpretation

Because the file is already constructed at the transition level, this notebook estimates a **cross-sectional regression on a transition outcome** for a single year pair. This is your **pair-specific DiD-style regression** for the Recruitment margin.


In [3]:
import pandas as pd
import numpy as np
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 150)


#EDIT THIS when you switch to a new pair of years
DATA_FILE = "/Users/vedikabaradwaj/Documents/pdi_pensions/morg_data/analytic samples/analytic_sample_0809_recruitment.csv"

#Main variable choices
DV = "joined_state_local"
TREATMENT_VAR = "illinois"
PAIR_VAR = "pair"
WEIGHT_VAR = "weight_t"

#Baseline controls
# If your file uses docc80_t instead of docc00_t, swap that one line below.
BASE_CONTROLS = [
    "age_t",
    "I(age_t**2)",
    "C(sex_t)",
    "C(race_t)",
    "C(grade92_t)",
    "np.log(earnwke_t)",
    "C(docc00_t)",
    "C(ind02_t)",
    "C(unionmme_t)",
]

#Robust SE choice
COV_TYPE = "HC1"


## Loading the Data

In [4]:
df = pd.read_csv(DATA_FILE, low_memory=False)

print("Shape:", df.shape)
print("\nPair values:")
print(df[PAIR_VAR].value_counts(dropna=False))

print("\nColumns:")
print(df.columns.tolist())

Shape: (18302, 205)

Pair values:
pair
2008-2009    18302
Name: count, dtype: int64

Columns:
['hhid', 'intmonth', 'hurespli_t', 'hrhtype_t', 'minsamp_x', 'hrlonglk_t', 'hrsample_t', 'hrhhid2_t', 'serial_t', 'hhnum', 'state', 'stfips_t', 'cbsafips_t', 'county_t', 'centcity_t', 'smsastat_t', 'icntcity_t', 'smsa04_t', 'relref95_t', 'age_t', 'spouse_t', 'sex_t', 'grade92_t', 'race_t', 'ethnic_t', 'lineno', 'famnum_t', 'pfamrel_t', 'marital_t', 'prpertyp_t', 'penatvty_t', 'pemntvty_t', 'pefntvty_t', 'prcitshp_t', 'prcitflg_t', 'peinusyr_t', 'selfproxy_t', 'lfsr94_t', 'absent94_t', 'uhourse_t', 'reason94_t', 'hourslw_t', 'laydur_t', 'dwrsn_t', 'why3594_t', 'prunedur_t', 'untype_t', 'ftpt94_t', 'class94_t', 'agri_t', 'eligible_t', 'otc_t', 'ernpdh2_t', 'paidhre_t', 'earnhre_t', 'earnwke_t', 'unionmme_t', 'unioncov_t', 'schenr_t', 'studftpt_t', 'schlvl_t', 'earnwt_t', 'weight_t', 'chldpres_t', 'ownchild_t', 'i25d_t', 'i25c_t', 'i25a_t', 'i25b_t', 'qstnum_t', 'occurnum_t', 'ged_t', 'gedhigr_t'

## Validating the Data (chekcing for missing columns and removing them)

In [5]:
required_cols = [DV, TREATMENT_VAR, PAIR_VAR, WEIGHT_VAR]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

print("All required columns are present.")


All required columns are present.


## Descriptive Checks

In [6]:
print("\nUnweighted mean of DV by Illinois:")
print(df.groupby(TREATMENT_VAR)[DV].mean())

def weighted_mean(g):
    x = g[DV]
    w = g[WEIGHT_VAR]
    mask = x.notna() & w.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(x[mask], weights=w[mask])

print("\nWeighted mean of DV by Illinois:")
print(df.groupby(TREATMENT_VAR).apply(weighted_mean))

if "state_t" in df.columns:
    print("\nUnweighted mean of DV by state_t:")
    print(df.groupby("state_t")[DV].mean())



Unweighted mean of DV by Illinois:
illinois
0    0.033084
1    0.034483
Name: joined_state_local, dtype: float64

Weighted mean of DV by Illinois:
illinois
0    0.033328
1    0.033812
dtype: float64

Unweighted mean of DV by state_t:
state_t
illinois        0.034483
indiana         0.034199
new_york        0.038237
pennsylvania    0.026269
Name: joined_state_local, dtype: float64


/var/folders/j8/n0941h8j0wj8kz_20gdz081m0000gn/T/ipykernel_71843/4070057134.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby(TREATMENT_VAR).apply(weighted_mean))


## Baseline specification

The default model in this notebook is a weighted Linear Probability Model:

$$Y_i = \alpha + \beta \, \text{Illinois}_i + X_i'\theta + \varepsilon_i$$

where:
- \(Y_i\) is `joined_state_local`
- the coefficient on `illinois` is the pair-specific Illinois-control difference for the **Recruitment** outcome

In [12]:
def build_formula(dv, controls=None):
    controls = controls or []
    rhs_terms = [TREATMENT_VAR] + controls
    rhs = " + ".join(rhs_terms)
    return f"{dv} ~ {rhs}"

formula = build_formula(DV, BASE_CONTROLS)
print(formula)


joined_state_local ~ illinois + age_t + C(sex_t) + C(race_t) + C(grade92_t)


In [13]:
def extract_needed_columns(controls):
    """
    Pull raw dataframe column names out of patsy-style terms like:
    C(x), I(x**2), np.log(x), C(x):C(z), etc.
    """
    cols = set()

    for c in controls:
        c_matches = re.findall(r"C\(([^)]+)\)", c)
        for m in c_matches:
            cols.add(m.strip())

        i_matches = re.findall(r"I\(([^)]+)\)", c)
        for expr in i_matches:
            var_matches = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr)
            for v in var_matches:
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)

        log_matches = re.findall(r"np\.log\(([^)]+)\)", c)
        for m in log_matches:
            cols.add(m.strip())

        if (
            not c.startswith("C(")
            and not c.startswith("I(")
            and not c.startswith("np.log(")
        ):
            var_matches = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", c)
            for v in var_matches:
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)

    return list(cols)


def run_weighted_lpm(df, dv, controls=None, weight_var=WEIGHT_VAR, cov_type=COV_TYPE):
    controls = controls or []
    data = df.copy()

    raw_controls = extract_needed_columns(controls)

    needed_cols = [dv, TREATMENT_VAR, weight_var] + raw_controls
    needed_cols = [c for c in needed_cols if c in data.columns]

    before = len(data)
    data = data.dropna(subset=needed_cols).copy()

    if "np.log(earnwke_t)" in controls and "earnwke_t" in data.columns:
        data = data[data["earnwke_t"] > 0].copy()

    after = len(data)
    print(f"Dropped {before - after:,} rows due to missing values; {after:,} rows remain.")

    formula = build_formula(dv, controls)
    print(f"Running formula: {formula}")

    model = smf.wls(formula=formula, data=data, weights=data[weight_var])
    result = model.fit(cov_type=cov_type)

    print(result.summary())
    return result

lpm_result = run_weighted_lpm(df, DV, BASE_CONTROLS)


Dropped 0 rows due to missing values; 18,302 rows remain.
                            WLS Regression Results                            
Dep. Variable:     joined_state_local   R-squared:                       0.006
Model:                            WLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     22.25
Date:                Sun, 05 Apr 2026   Prob (F-statistic):          2.51e-111
Time:                        17:59:14   Log-Likelihood:                   -inf
No. Observations:               18302   AIC:                               inf
Df Residuals:                   18273   BIC:                               inf
Df Model:                          28                                         
Covariance Type:                  HC1                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/linear_model.py:806: RuntimeWarning: divide by zero encountered in log
  llf += 0.5 * np.sum(np.log(self.weights))


In [15]:

result_table = pd.DataFrame({
    "term": ["illinois"],
    "coef": [lpm_result.params.get("illinois", np.nan)],
    "std_err": [lpm_result.bse.get("illinois", np.nan)],
    "p_value": [lpm_result.pvalues.get("illinois", np.nan)],
    "nobs": [int(lpm_result.nobs)]
})

result_table


,term,coef,std_err,p_value,nobs
0,illinois,0.000143,0.003072,0.962875,18302
